# TopK SAE on a vision transformer

This notebook trains a sparse autoencoder on the residual stream of a
pretrained **ViT-Tiny** (timm, ImageNet-1k classifier, d = 192) using
**Imagenette** (a 10-class ImageNet subset, ~9.5k training images), then
analyses what it learned.

Workflow:

1. **Load a pretrained model and a dataset.** The model stays frozen; the
   SAE explains its activations, so there is no model training here.
2. **Harvest activations** at one layer with a forward hook, drop the CLS
   token, shuffle across images, and **cache them to disk** (safetensors
   shards). Caching pays off when you train for several epochs or sweep
   hyperparameters; for a single pass you can train straight from the
   `RowStream`.
3. **Train the SAE** on the cached activations.
4. **Evaluate**: reconstruction (FVE) and the effect of splicing the SAE
   into the model (KL to the clean predictions, top-1 agreement).
5. Optionally **KL fine-tune** the spliced SAE (Karvonen, 2025).
6. **Save / load** the SAE as a single safetensors file.
7. **Analyse the dictionary**: density, redundancy, a 2-D map, class effects.
8. **Inspect latents**: top-activating images with patch heatmaps.
9. **Attribute** a prediction to latents and patches.

Install: `uv sync --extra notebooks` (or `uv pip install -e ".[notebooks]"`).

Set `SAE_SMOKE_TEST=1` to run every cell offline with a random-weight model
and fake images (for CI); the results are then meaningless.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from innards import RowLoader, RowStream, write_rows
from innards.sae import (
    KLFinetuneConfig,
    SAETrainer,
    SAE,
    TrainerConfig,
)
import torch.nn.functional as F

from innards.analysis import geometry, interp, plot
from innards.sae import analysis as A
from innards.sae import viz

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AUTOCAST = torch.bfloat16 if DEVICE == "cuda" else None

MODEL_NAME = "vit_tiny_patch16_224.augreg_in21k_ft_in1k"
LAYER = 6 # residual stream after block 6 of 12
DATA_DIR = Path("data")
CACHE_DIR = Path("activations") / f"{MODEL_NAME}_block{LAYER}"
OUT_DIR = Path("checkpoints") / f"{MODEL_NAME}_block{LAYER}"
BATCH_IMAGES = 64
NUM_WORKERS = 4
print(f"device={DEVICE}")

## 1. Model and data

`timm` gives the matching preprocessing via `resolve_data_config`.
torchvision downloads Imagenette (160 px version, ~100 MB) on first use.

In [ ]:
import timm
import torchvision

model = timm.create_model(MODEL_NAME, pretrained=True)
data_cfg = timm.data.resolve_data_config({}, model=model)
transform = timm.data.create_transform(**data_cfg)
MEAN, STD = data_cfg["mean"], data_cfg["std"]
download = not (DATA_DIR / "imagenette2-160").exists()
train_ds = torchvision.datasets.Imagenette(DATA_DIR, "train", "160px", download=download, transform=transform)
val_ds = torchvision.datasets.Imagenette(DATA_DIR, "val", "160px", transform=transform)

model = model.eval().to(DEVICE)
layer = model.blocks[LAYER]
PREFIX = tuple(range(model.num_prefix_tokens)) # CLS (+ register tokens, if any)
print(f"d_model={model.embed_dim}, prefix tokens={PREFIX}, train images={len(train_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_IMAGES, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_IMAGES, shuffle=False, num_workers=NUM_WORKERS)

## 2. Harvest and cache activations

The store runs the model up to `blocks[LAYER]` (the rest of the forward pass
is skipped), drops the CLS token, and shuffles tokens from many images in a
buffer. `write_rows` stores them as bf16 shards: one pass over Imagenette is
~1.9M tokens, about 0.7 GB.

In [ ]:
stream = RowStream(model, train_loader, layer, exclude=PREFIX, device=DEVICE, dtype=torch.bfloat16,
                   autocast_dtype=AUTOCAST)
if not (CACHE_DIR / "meta.json").exists():
    write_rows(stream, CACHE_DIR, dtype=torch.bfloat16,
               metadata={"model": MODEL_NAME, "layer": f"blocks.{LAYER}", "exclude": list(PREFIX)})
train_acts = RowLoader(CACHE_DIR, device=DEVICE)
print(f"cached tokens: {train_acts.n_rows:,}")

## 3. Train the SAE

8x expansion (3072 latents), k = 32, BatchTopK. Five epochs over the cache
is ~9M tokens: enough for a usable SAE on this small model, though more
data (e.g. ImageNet) gives cleaner latents. Add `"wandb"` or `"mlflow"` to
`loggers` if you use them; TensorBoard logs go to `runs/`.

In [ ]:
D_MODEL = model.embed_dim
sae = SAE(d_in=D_MODEL, d_sae=8 * D_MODEL, k=16, activation="batchtopk")
trainer = SAETrainer(sae, TrainerConfig(
    num_epochs=5,
    loggers=("tensorboard",),
    run_name=f"vit-tiny-block{LAYER}",
    output_dir=str(OUT_DIR),
    log_every=200,
))
trainer.train(train_acts)

## 4. Evaluate

Reconstruction on held-out images, then the downstream view: splice the SAE
into the ViT and compare predictions with the clean model.

In [ ]:
val_stream = RowStream(model, val_loader, layer, exclude=PREFIX, device=DEVICE, dtype=torch.bfloat16,
                       autocast_dtype=AUTOCAST)
kl_cfg = KLFinetuneConfig(steps=300, exclude=PREFIX, log_every=50)
print(trainer.evaluate(val_stream))
print(trainer.evaluate_downstream(model, layer, val_loader, kl_cfg))

## 5. KL fine-tuning (optional)

A short end-to-end fine-tune on KL + reconstruction lowers the downstream
KL at a small reconstruction cost.

In [ ]:
RUN_KL_FINETUNE = True
if RUN_KL_FINETUNE:
    trainer.finetune_kl(model, layer, train_loader, kl_cfg)
    print(trainer.evaluate(val_stream))
    print(trainer.evaluate_downstream(model, layer, val_loader, kl_cfg))

## 6. Save and load

The trainer already wrote `sae.safetensors` (and `sae_kl.safetensors`). The
file holds the weights with the input normalisation folded in and the
config in its metadata, so loading needs nothing else.

In [ ]:
path = trainer.save(OUT_DIR / "sae_final.safetensors", model=MODEL_NAME, hook=f"blocks.{LAYER}")
sae = SAE.load(path, device=DEVICE)
print(SAE.read_metadata(path)["hook"], sae.cfg)

## 7. The dictionary as a whole

One pass over the validation set gives firing frequencies and the
top-activating patches of every latent. Then: how redundant are the atoms,
how do they spread out, and which classes does each one push towards?

In [ ]:
res = A.analyze_dataset(sae, model, layer, val_loader, exclude=PREFIX)
atoms = F.normalize(sae.decoder_weight().detach(), dim=-1)  # the dictionary as unit directions
max_cos, _ = geometry.nearest(atoms, k=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
plot.histogram(res.stats.frequency, log10=True, xlabel="log10 firing frequency", ylabel="latents",
               title="latent density", ax=axes[0])
plot.histogram(max_cos, xlabel="max cosine similarity", ylabel="latents", title="decoder redundancy", ax=axes[1])
plot.embedding(atoms, method="pca", values=torch.log10(res.stats.frequency + 1e-6),  # "umap" with the umap extra
               values_label="log10 frequency", title="dictionary map", ax=axes[2])
plt.show()
print(f"dead: {len(res.stats.dead)}, dense (>10%): {len(res.stats.dense())}")

Class effects project each decoder atom onto the classifier head. This is
exact only at the last layer (and ignores the final LayerNorm), so at a
middle layer read it as a hint, not a measurement.

In [ ]:
from timm.data import ImageNetInfo

info = ImageNetInfo()
class_name = lambda i: info.index_to_description(i).split(",")[0]

alive = (res.stats.frequency > 1e-4).nonzero().squeeze(-1)
latents = alive[torch.randperm(len(alive), generator=torch.Generator().manual_seed(0))[:4]].tolist()
top_v, top_i, bot_v, bot_i = interp.top_outputs(interp.readout_effect(atoms[latents], model.head.weight), n=5)
fig, axes = plt.subplots(1, len(latents), figsize=(5 * len(latents), 3.8), layout="constrained")
for j, lat in enumerate(latents):
    ids = torch.cat([top_i[j], bot_i[j].flip(0)])
    plot.bars(torch.cat([top_v[j], bot_v[j].flip(0)]), [class_name(int(i)) for i in ids],
              xlabel="direct effect", title=f"latent {lat}", ax=axes[j])
plt.show()

## 8. Latent dashboards: top-activating images

For each latent: the images containing its strongest activations, with the
latent's activation over the patch grid overlaid (one colour scale per
latent).

In [ ]:
from innards.extract import layer_tokens


def show_latent(latent: int, n: int = 8):
    examples = res.top.unique_examples(latent)[:n]
    if not examples:
        print(f"latent {latent} never fired")
        return
    imgs = torch.stack([val_ds[e][0] for _, e, _ in examples]).to(DEVICE)
    with torch.no_grad():
        tokens = layer_tokens(model, imgs, layer)
    maps = interp.spatial_map(tokens, A.latent_fn(sae, latent), tuple(model.patch_embed.grid_size), exclude=PREFIX)
    labels = [class_name(int(val_ds[e][1])) for _, e, _ in examples]
    titles = [f"{v:.2f} {lab}" for (v, _, _), lab in zip(examples, labels)]
    plot.images(imgs.cpu(), maps=maps.cpu(), titles=titles, mean=MEAN, std=STD,
                title=f"latent {latent} (freq {res.stats.frequency[latent]:.2%})")
    plt.show()


for lat in latents[:3]:
    show_latent(lat)

Neighbouring atoms often split one concept (e.g. a texture at different
scales). Compare a latent with its nearest neighbours in the dictionary.

In [ ]:
sims, neighbours = geometry.nearest(atoms, latents[0], k=5)
group = [latents[0], *neighbours.tolist()]
plot.similarity_matrix(atoms[group] @ atoms[group].T, labels=[str(l) for l in group], signed=True,
                       cbar_label="cosine similarity")
plt.show()

## 9. Attribution: which latents drive a prediction?

Splice the SAE (with its error term, so the output is unchanged) and
attribute the predicted-class logit to every latent at every patch
(attribution patching). The bars show the latents with the largest total
effect; the heatmaps show where in the image they contribute, plus a
pixel-level gradient x input map of the top latent.

In [ ]:
image, label = val_ds[0]
x = image[None].to(DEVICE)
with torch.no_grad():
    pred = model(x).argmax(-1).item()

attr = A.latent_attribution(sae, model, layer, x, lambda out, _: out[:, pred], exclude=PREFIX)
values, top = attr.top_latents(0, n=8)
plot.bars(torch.cat([values, attr.error_attribution[0].sum()[None]]),
          [f"#{int(l)}" for l in top] + ["SAE error"], xlabel="attribution", title=f"logit of '{class_name(pred)}'")
plt.show()

per_patch = attr.attribution[0, attr.keep[0]][:, top[:3]].T.cpu().reshape(3, *model.patch_embed.grid_size)
saliency = interp.input_attribution(model, x, f"blocks.{LAYER}", A.latent_fn(sae, int(top[0]), pre=True),
                                    exclude=PREFIX)
img = plot.to_display_image(image, MEAN, STD)
fig, axes = plt.subplots(1, 5, figsize=(16, 3.4), layout="constrained")
axes[0].imshow(img); axes[0].set_title(f"pred: {class_name(pred)}"); axes[0].axis("off")
for j in range(3):
    plot.overlay_heatmap(img, per_patch[j], ax=axes[j + 1], signed=True)
    axes[j + 1].set_title(f"latent {int(top[j])}: {values[j]:.2f}")
axes[4].imshow(saliency[0].abs().sum(0).cpu(), cmap="inferno"); axes[4].axis("off")
axes[4].set_title(f"pixels -> latent {int(top[0])}")
plt.show()